# T2.1 — Autoencoder retrain (hardened) for the Stage-A anomaly screen

The first AE **collapsed** (reconstruction std ≈ 0.007, corr(input,recon) ≈ 0.10 — it output a
constant mid-grey). Root cause: too little capacity (16→32 ch), only 30 epochs, no collapse
guard, no recorded AUROC. This notebook fixes all of that: deeper AE (32→64→128→256, real
40×40 bottleneck), SSIM+L1, 100 epochs, cosine LR, sonar-safe augmentation, a **per-epoch
recon-std collapse guard that refuses to export**, and **tile-level AUROC** (T2.1's real gate).

**Before running (data is NOT on GitHub — gitignored):** upload to your Google Drive:
1. **`T2.1_anomaly_dataset.zip`** (~165 MB) or the full `Samudra_Prahari_Dataset_v2/` — supplies
   the object crops used as AUROC positives (needed either way).
2. **Recommended:** the AI4Shipwrecks **`extras/terrain`** folder (real object-free sonar) — see
   the "REAL terrain" cell below. Without it the AE trains on synthetic seabed and its anomaly
   score is sim-to-real **confounded** (0.51 mean / 0.77 max AUROC); with it, the number is honest.

Then run top-to-bottom on a GPU runtime. Target: recon-std well above 0.02 and AUROC ≥ 0.80.

In [ ]:
# 0. Setup — clone YOUR repo, install deps
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!git clone https://github.com/ASugandhan/SamudraPrahari.git || (cd SamudraPrahari && git pull)
%cd SamudraPrahari
!pip install -q -e '.[models]' pytorch-msssim scikit-learn

In [ ]:
# 1. Data — images aren't on GitHub. Point DRIVE at where the data is in your Drive.
#    Handles ANY of: extracted T2.1_anomaly_dataset FOLDER, the .zip, or the full dataset.
import os, glob, zipfile, shutil, subprocess
from pathlib import Path

DRIVE = '/content/drive/MyDrive'      # <-- folder in your Drive that holds the data

def _sh(*cmd):
    print('$', ' '.join(cmd)); subprocess.run(cmd, check=True)

def _t21_paths():
    d = 'T2.1_anomaly_dataset'
    return (f'{d}/normal_seabed/train', f'{d}/normal_seabed/val',
            f'{d}/normal_seabed/test/*.jpg', f'{d}/object_eval_set/*.jpg')

def prep_data():
    ok = lambda: Path('T2.1_anomaly_dataset/normal_seabed/train').exists()
    # (A0) already prepared locally
    if ok(): return _t21_paths()
    # (A1) extracted FOLDER somewhere in Drive -> copy it in (fast local reads for training)
    for cand in glob.glob(f'{DRIVE}/**/T2.1_anomaly_dataset', recursive=True):
        if Path(cand, 'normal_seabed/train').exists():
            print('copying', cand, '-> ./T2.1_anomaly_dataset'); shutil.copytree(cand, 'T2.1_anomaly_dataset')
            break
    if ok(): return _t21_paths()
    # (A2) the .zip in Drive -> unzip
    for z in glob.glob(f'{DRIVE}/**/T2.1_anomaly_dataset.zip', recursive=True):
        print('unzipping', z); zipfile.ZipFile(z).extractall('.'); break
    if ok(): return _t21_paths()
    # (B) full dataset -> index + materialise the leak-free re-split
    if not Path('Samudra_Prahari_Dataset_v2').exists():
        for c in glob.glob(f'{DRIVE}/**/Samudra_Prahari_Dataset_v2', recursive=True):
            os.symlink(c, 'Samudra_Prahari_Dataset_v2'); break
    if Path('Samudra_Prahari_Dataset_v2').exists():
        _sh('python', 'scripts/data/index_samudra_v2.py')
        _sh('python', 'scripts/data/materialize_resplit.py')
        return ('_resplit/anomaly/images/train', '_resplit/anomaly/images/val',
                '_resplit/anomaly/images/test/*.jpg', '_resplit/detection/images/test/*.jpg')
    raise FileNotFoundError(
        f'No data under {DRIVE}. Put the T2.1_anomaly_dataset folder/zip OR '
        'Samudra_Prahari_Dataset_v2/ there (or fix the DRIVE path above).')

TRAIN_DIR, VAL_DIR, NORMAL_TEST_GLOB, OBJECT_TEST_GLOB = prep_data()
n_train = len(glob.glob(TRAIN_DIR + '/*.jpg'))
assert n_train > 0, f'no training tiles in {TRAIN_DIR}'
print(f'train {n_train} | normal-test {len(glob.glob(NORMAL_TEST_GLOB))} | '
      f'object-test {len(glob.glob(OBJECT_TEST_GLOB))}')

In [ ]:
# 2. Config
import torch
SIZE, EPOCHS, BATCH, LR = 640, 100, 8, 1e-3   # batch 8 is T4-safe at 640px; raise if you have VRAM
CH = (32, 64, 128, 256)          # encoder channels (real 40x40 bottleneck)
COLLAPSE_STD = 0.02              # refuse to export if val recon std < this
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## Recommended: train on REAL AI4Shipwrecks terrain (removes the sim-to-real confound)
The synthetic-trained AE detects *real-vs-synthetic*, not *object-vs-none* (AUROC 0.51 mean /
0.77 max — confounded). Training on REAL object-free terrain fixes this.

**One-time download (manual — Deep Blue requires accepting terms, so it can't be wget'd):**
1. Open **https://deepblue.lib.umich.edu/data/concern/data_sets/8623hz41x**
2. Download the dataset; inside it is **`extras/terrain/`** — object-free real sonar PNGs.
3. Put that `terrain` folder anywhere in your Google Drive.

The next cell auto-detects it, tiles it to 640px (grouped by survey, leak-free), and switches
BOTH training and the AUROC eval to **real-terrain vs real-objects**. If no terrain is found it
falls back to the synthetic seabed (confounded) with a loud warning.

In [ ]:
# 2b. Use REAL terrain if present (recommended), else fall back to synthetic with a warning
import os, glob, shutil, cv2
from pathlib import Path
from samudra.common.splits import split_by_group

def _find_terrain():
    hits = []
    for pat in (f'{DRIVE}/**/extras/terrain/**/*.png', f'{DRIVE}/**/terrain/**/*.png'):
        hits += glob.glob(pat, recursive=True)
    return sorted(set(hits))

def _tile_terrain(pngs, out, size=SIZE, stride=None, min_fill=0.5):
    stride = stride or size // 2
    Path(out).mkdir(parents=True, exist_ok=True)
    groups = {}
    for p in pngs:                                   # group tiles by survey -> leak-free split (R5)
        g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if g is None: continue
        H, W = g.shape; stem = Path(p).stem; groups.setdefault(stem, [])
        for y in range(0, max(1, H - size + 1), stride):
            for x in range(0, max(1, W - size + 1), stride):
                c = g[y:y+size, x:x+size]
                if c.shape != (size, size): c = cv2.resize(c, (size, size))
                if (c > 5).mean() < min_fill: continue    # skip mostly-black water column / padding
                fn = f'{stem}__{y}_{x}.png'; cv2.imwrite(f'{out}/{fn}', c); groups[stem].append(fn)
    return {k: v for k, v in groups.items() if v}

terrain = _find_terrain()
if terrain:
    print(f'REAL terrain: {len(terrain)} survey PNGs -> tiling to {SIZE}px ...')
    groups = _tile_terrain(terrain, 'ai4_terrain/_all')
    for s in ('train', 'val', 'test'): Path(f'ai4_terrain/{s}').mkdir(parents=True, exist_ok=True)
    allfiles = [(stem, fn) for stem, fns in groups.items() for fn in fns]
    assign = split_by_group(list(groups), seed=1337) if len(groups) >= 3 else None
    grouped_ok = assign is not None and set(assign.values()) == {'train', 'val', 'test'}
    if grouped_ok:                                    # split whole surveys (no leakage)
        for stem, fn in allfiles: shutil.move(f'ai4_terrain/_all/{fn}', f'ai4_terrain/{assign[stem]}/{fn}')
        print(f'  grouped split by survey ({len(groups)} surveys) — leak-free.')
    else:                                             # too few surveys -> per-tile split (note leakage)
        why = 'fewer than 3 surveys' if assign is None else 'grouped split left a split empty'
        print(f'  {why} -> per-tile split (adjacent-tile overlap leakage; noted, not leak-free).')
        import random; rng = random.Random(1337); rng.shuffle(allfiles)
        n = len(allfiles); c1, c2 = int(.7 * n), int(.85 * n)
        for i, (_, fn) in enumerate(allfiles):
            s = 'train' if i < c1 else 'val' if i < c2 else 'test'
            shutil.move(f'ai4_terrain/_all/{fn}', f'ai4_terrain/{s}/{fn}')
    TRAIN_DIR, VAL_DIR, NORMAL_TEST_GLOB = 'ai4_terrain/train', 'ai4_terrain/val', 'ai4_terrain/test/*.png'
    # positives = REAL objects only (ai4_/sctd_) -> AUROC is real-vs-real (unconfounded)
    hits = glob.glob(OBJECT_TEST_GLOB)
    obj_dir = str(Path(hits[0]).parent) if hits else 'T2.1_anomaly_dataset/object_eval_set'
    Path('real_objects').mkdir(exist_ok=True)
    for pre in ('ai4_', 'sctd_'):
        for f in glob.glob(f'{obj_dir}/{pre}*.jpg'):
            d = f'real_objects/{Path(f).name}'
            if not Path(d).exists(): os.symlink(os.path.abspath(f), d)
    OBJECT_TEST_GLOB = 'real_objects/*.jpg'
    print(f'MODE = REAL terrain | train {len(glob.glob(TRAIN_DIR+"/*.png"))} tiles | '
          f'test-neg {len(glob.glob(NORMAL_TEST_GLOB))} | real-obj-pos {len(glob.glob(OBJECT_TEST_GLOB))}')
    print('-> AUROC will be real-terrain vs real-objects (the honest, unconfounded number).')
else:
    print('NO AI4Shipwrecks terrain found under', DRIVE, '-> FALLING BACK to SYNTHETIC seabed.')
    print('Results will be sim-to-real CONFOUNDED. Download extras/terrain from Deep Blue to fix.')

In [ ]:
# 3. Dataset + sonar-safe augmentation (seabed is orientation-agnostic -> flips/rot90 safe)
import numpy as np, cv2
from torch.utils.data import Dataset, DataLoader

def load_gray01(path, size=SIZE):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
    return g.astype(np.float32) / 255.0

class SeabedTiles(Dataset):
    def __init__(self, folder, augment):
        self.paths = sorted(glob.glob(folder + '/*.jpg')) + sorted(glob.glob(folder + '/*.png'))
        self.augment = augment
        assert self.paths, f'no images in {folder}'
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = load_gray01(self.paths[i])
        if self.augment:
            if np.random.rand() < 0.5: x = x[:, ::-1]
            if np.random.rand() < 0.5: x = x[::-1, :]
            x = np.rot90(x, k=np.random.randint(4))
            x = np.clip(x * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.03, 0.03), 0, 1)
        return torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[None]

train_dl = DataLoader(SeabedTiles(TRAIN_DIR, True),  batch_size=BATCH, shuffle=True, drop_last=True)
val_dl   = DataLoader(SeabedTiles(VAL_DIR,   False), batch_size=BATCH, shuffle=False)
print('train tiles:', len(train_dl.dataset), '| val tiles:', len(val_dl.dataset))

In [ ]:
# 4. Model — conv autoencoder with a real bottleneck (no more collapse)
import torch.nn as nn

class ConvAE(nn.Module):
    def __init__(self, ch=CH):
        super().__init__()
        def enc(i, o): return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        def dec(i, o, last=False):
            L = [nn.ConvTranspose2d(i, o, 4, 2, 1)]
            L += [nn.Sigmoid()] if last else [nn.BatchNorm2d(o), nn.ReLU(True)]
            return nn.Sequential(*L)
        c1, c2, c3, c4 = ch
        self.enc = nn.Sequential(enc(1, c1), enc(c1, c2), enc(c2, c3), enc(c3, c4))          # 640->40
        self.dec = nn.Sequential(dec(c4, c3), dec(c3, c2), dec(c2, c1), dec(c1, 1, True))    # 40->640
    def forward(self, x): return self.dec(self.enc(x))

model = ConvAE().to(DEVICE)
print(round(sum(p.numel() for p in model.parameters()) / 1e6, 2), 'M params')

In [ ]:
# 5. Train — SSIM+L1, cosine LR, per-epoch reconstruction-std logging
import torch.nn.functional as F
from pytorch_msssim import ssim

def loss_fn(recon, x):
    return 0.5 * (1 - ssim(recon, x, data_range=1.0, size_average=True)) + 0.5 * F.l1_loss(recon, x)

opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

@torch.no_grad()
def val_stats():
    model.eval(); losses, stds = [], []
    for xb in val_dl:
        xb = xb.to(DEVICE); rec = model(xb)
        losses.append(loss_fn(rec, xb).item())
        stds.append(rec.std(dim=(2, 3)).mean().item())   # ~0 => collapsed to constant
    return float(np.mean(losses)), float(np.mean(stds))

hist = {'train': [], 'val': [], 'recon_std': []}
for ep in range(EPOCHS):
    model.train(); tl = []
    for xb in train_dl:
        xb = xb.to(DEVICE); opt.zero_grad()
        loss = loss_fn(model(xb), xb); loss.backward(); opt.step(); tl.append(loss.item())
    sched.step()
    vl, vstd = val_stats()
    hist['train'].append(float(np.mean(tl))); hist['val'].append(vl); hist['recon_std'].append(vstd)
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f'ep {ep:3d}  train {np.mean(tl):.4f}  val {vl:.4f}  recon_std {vstd:.4f}'
              + ('   <-- COLLAPSE WARNING' if vstd < COLLAPSE_STD else ''))

In [ ]:
# 6. Loss curves + hard collapse guard
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
Path('out').mkdir(exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist['train'], label='train'); ax[0].plot(hist['val'], label='val')
ax[0].set_title('SSIM+L1 loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].axhline(COLLAPSE_STD, color='r', ls='--', label='collapse floor')
ax[1].plot(hist['recon_std'], label='recon std'); ax[1].set_title('reconstruction std'); ax[1].legend()
fig.savefig('out/t21_loss.png', dpi=110); print('saved out/t21_loss.png')

final_std = hist['recon_std'][-1]
assert final_std >= COLLAPSE_STD, (f'COLLAPSED: recon_std {final_std:.4f} < {COLLAPSE_STD}. '
    'Do NOT export. Raise EPOCHS to 200 / CH to (48,96,192,384) / lower LR to 5e-4 and retrain.')
print(f'OK — recon_std {final_std:.4f} (not collapsed)')

In [ ]:
# 7. Tile-level AUROC — object-free (neg) vs object crops (pos). T2.1's real acceptance gate.
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def anomaly_score(path):
    x = torch.from_numpy(load_gray01(path))[None, None].to(DEVICE)
    return torch.abs(x - model(x)).mean().item()

model.eval()
neg = [anomaly_score(p) for p in glob.glob(NORMAL_TEST_GLOB)]
pos = [anomaly_score(p) for p in glob.glob(OBJECT_TEST_GLOB)]
auroc = roc_auc_score([0] * len(neg) + [1] * len(pos), neg + pos)
print(f'tile AUROC (object vs no-object) = {auroc:.3f}  (target >= 0.80)  '
      f'neg={len(neg)} pos={len(pos)}  mean neg {np.mean(neg):.3f} vs pos {np.mean(pos):.3f}')

In [ ]:
# 8. Export ONNX as ONE self-contained file (no external .data to lose) + register
import json, onnx, os
model.eval().cpu()
torch.onnx.export(model, torch.randn(1, 1, SIZE, SIZE), 'models/stage_a_ae.onnx', opset_version=13,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'b'}, 'output': {0: 'b'}})
m = onnx.load('models/stage_a_ae.onnx')          # pulls any external weights into memory
m.ir_version = min(m.ir_version, 10)             # onnxruntime here supports ir <= 13
onnx.save_model(m, 'models/stage_a_ae.onnx', save_as_external_data=False)  # single file, weights inline
if os.path.exists('models/stage_a_ae.onnx.data'): os.remove('models/stage_a_ae.onnx.data')

json.dump({'task': 'T2.1', 'test_loss': round(hist['val'][-1], 4), 'epochs': EPOCHS,
           'recon_std': round(final_std, 4), 'auroc': round(float(auroc), 4)},
          open('out/metrics_t21.json', 'w'), indent=2)
!python scripts/register_model.py --onnx models/stage_a_ae.onnx --name stage_a_ae --trained-on 'AnomalyScreen normal_seabed (synthetic, object-free)' --metrics out/metrics_t21.json
!cp models/stage_a_ae.onnx* models/registry.json out/metrics_t21.json /content/drive/MyDrive/ 2>/dev/null; echo copied-to-Drive

## After it runs
1. Check the printout: **recon_std well above 0.02** and **AUROC ≥ 0.80**.
2. From your Drive, copy back into the repo: `models/stage_a_ae.onnx` (+ `.data` if present),
   `models/registry.json`, `out/metrics_t21.json`. Commit + push.
3. Locally: `python scripts/eval/stage_a_recall.py --split test` — the recall curve is now
   meaningful and **T2.2 unblocks**.

Still collapsing? Bump `EPOCHS=200`, `CH=(48,96,192,384)`, `LR=5e-4` (cell 2) and rerun.